# HDB Resale Flat Prices - ETL Pipeline

## Overview
This Jupyter Notebook implements a comprehensive ETL (Extract, Transform, Load) pipeline for HDB resale flat prices data from 2012-2016. The pipeline includes:
- Data extraction and consolidation from multiple CSV files
- Data quality profiling and validation
- Duplicate handling and anomaly detection
- Data transformation with Resale Identifier creation
- Secure hashing of identifiers
- Generation of cleaned, transformed, and failed datasets

## Dataset Period: January 2012 - December 2016

## Section 1: Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import hashlib
import os
from datetime import datetime, timedelta
from pathlib import Path
import warnings
import re
from typing import Tuple, List, Dict
import shutil

# Set display options for better visibility
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("✓ All required libraries imported successfully")
print(f"Execution timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✓ All required libraries imported successfully
Execution timestamp: 2026-07-19 16:29:30


## Section 2: Setup and Configuration

In [2]:
# Define directory paths
BASE_DIR = Path.cwd()
RAW_DIR = BASE_DIR / "raw"
OUTPUT_DIR = BASE_DIR / "output"

# Create output directories if they don't exist
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "raw").mkdir(exist_ok=True)
(OUTPUT_DIR / "cleaned").mkdir(exist_ok=True)
(OUTPUT_DIR / "transformed").mkdir(exist_ok=True)
(OUTPUT_DIR / "failed").mkdir(exist_ok=True)
(OUTPUT_DIR / "hashed").mkdir(exist_ok=True)

print(f"Base Directory: {BASE_DIR}")
print(f"Raw Data Directory: {RAW_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")

# Configuration constants
HDB_LEASE_YEARS = 99
REFERENCE_DATE = datetime.now()  # Today's date for lease calculation
PRICE_OUTLIER_THRESHOLD_IQR = 1.5  # IQR multiplier for outlier detection
PRICE_OUTLIER_THRESHOLD_ZSCORE = 3  # Z-score threshold for outlier detection

print(f"\n✓ Configuration set:")
print(f"  - HDB Lease Years: {HDB_LEASE_YEARS}")
print(f"  - Reference Date: {REFERENCE_DATE.strftime('%Y-%m-%d')}")
print(f"  - Price Outlier IQR Threshold: {PRICE_OUTLIER_THRESHOLD_IQR}")
print(f"  - Price Outlier Z-Score Threshold: {PRICE_OUTLIER_THRESHOLD_ZSCORE}")

Base Directory: /Users/sookyinchoong/HDB-ETL/part1
Raw Data Directory: /Users/sookyinchoong/HDB-ETL/part1/raw
Output Directory: /Users/sookyinchoong/HDB-ETL/part1/output

✓ Configuration set:
  - HDB Lease Years: 99
  - Reference Date: 2026-07-19
  - Price Outlier IQR Threshold: 1.5
  - Price Outlier Z-Score Threshold: 3


## Section 3: Dataset Extraction and Loading

This section loads all CSV files from the raw folder and combines them into a single master dataset. 
All attributes from all files are preserved in the master dataset.

In [3]:
def load_raw_datasets(raw_dir: Path) -> Tuple[pd.DataFrame, List[str]]:
    """
    Load all CSV files from the raw directory and combine them into a single master dataset.
    
    Args:
        raw_dir: Path to the directory containing raw CSV files
        
    Returns:
        Tuple of (combined_dataframe, list_of_file_names_loaded)
    """
    csv_files = sorted(raw_dir.glob("*.csv"))
    dataframes = []
    file_names = []
    
    print(f"Loading {len(csv_files)} CSV files from {raw_dir}...\n")
    
    for file in csv_files:
        try:
            df = pd.read_csv(file)
            dataframes.append(df)
            file_names.append(file.name)
            print(f"✓ Loaded: {file.name} ({len(df)} rows, {len(df.columns)} columns)")
        except Exception as e:
            print(f"✗ Error loading {file.name}: {str(e)}")
    
    # Combine all dataframes, aligning columns
    master_df = pd.concat(dataframes, ignore_index=True, sort=False)
    
    # Fill NaN values for columns that don't exist in some files
    # remaining_lease will be NaN for older datasets without this field
    
    print(f"\n{'='*70}")
    print(f"Master Dataset Summary:")
    print(f"{'='*70}")
    print(f"Total Rows: {len(master_df)}")
    print(f"Total Columns: {len(master_df.columns)}")
    print(f"Columns: {list(master_df.columns)}")
    
    return master_df, file_names

# Load all raw datasets
raw_master_df, loaded_files = load_raw_datasets(RAW_DIR)

Loading 5 CSV files from /Users/sookyinchoong/HDB-ETL/part1/raw...

✓ Loaded: Resale Flat Prices (Based on Approval Date), 1990 - 1999.csv (287196 rows, 10 columns)


✓ Loaded: Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv (369651 rows, 10 columns)
✓ Loaded: Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv (37153 rows, 11 columns)
✓ Loaded: Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv (52203 rows, 10 columns)


✓ Loaded: Resale flat prices based on registration date from Jan-2017 onwards.csv (235808 rows, 11 columns)

Master Dataset Summary:
Total Rows: 982011
Total Columns: 11
Columns: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price', 'remaining_lease']


In [4]:
# Display basic statistics of raw master dataset
print("\n" + "="*70)
print("RAW MASTER DATASET - DATA TYPES")
print("="*70)
print(raw_master_df.dtypes)

print("\n" + "="*70)
print("SAMPLE RECORDS (First 5 rows)")
print("="*70)
print(raw_master_df.head())

print("\n" + "="*70)
print("MISSING VALUES BY COLUMN")
print("="*70)
missing_data = raw_master_df.isnull().sum()
missing_pct = (raw_master_df.isnull().sum() / len(raw_master_df)) * 100
missing_df = pd.DataFrame({
    'Column': missing_data.index,
    'Missing_Count': missing_data.values,
    'Missing_Percentage': missing_pct.values
})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)
print(missing_df if len(missing_df) > 0 else "No missing values found")

# Save raw data copy to output directory
raw_copy_path = OUTPUT_DIR / "raw" / "master_raw_data.csv"
raw_master_df.to_csv(raw_copy_path, index=False)
print(f"\n✓ Raw master dataset saved to: {raw_copy_path}")


RAW MASTER DATASET - DATA TYPES
month                   object
town                    object
flat_type               object
block                   object
street_name             object
storey_range            object
floor_area_sqm         float64
flat_model              object
lease_commence_date      int64
resale_price           float64
remaining_lease         object
dtype: object

SAMPLE RECORDS (First 5 rows)
     month        town flat_type block       street_name storey_range  \
0  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     10 TO 12   
1  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     04 TO 06   
2  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     10 TO 12   
3  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     07 TO 09   
4  1990-01  ANG MO KIO    3 ROOM   216  ANG MO KIO AVE 1     04 TO 06   

   floor_area_sqm      flat_model  lease_commence_date  resale_price  \
0            31.0        IMPROVED                 1977        9000.0  

             Column  Missing_Count  Missing_Percentage
10  remaining_lease         709050           72.203876



✓ Raw master dataset saved to: /Users/sookyinchoong/HDB-ETL/part1/output/raw/master_raw_data.csv


## Section 4: Data Profiling

This section performs comprehensive data profiling on the master dataset, generating statistics including 
data types, missing values, distribution analysis, and summary statistics for each column.

In [5]:
def perform_data_profiling(df: pd.DataFrame) -> Dict:
    """
    Perform comprehensive data profiling on the dataset.
    
    Args:
        df: Input dataframe
        
    Returns:
        Dictionary containing profiling results
    """
    profile = {
        'total_rows': len(df),
        'total_columns': len(df.columns),
        'columns': df.columns.tolist(),
        'dtypes': df.dtypes.to_dict(),
        'duplicates': df.duplicated().sum(),
        'missing_values': df.isnull().sum().to_dict(),
        'missing_percentage': (df.isnull().sum() / len(df) * 100).to_dict(),
    }
    
    return profile

# Perform data profiling
profile = perform_data_profiling(raw_master_df)

print("="*70)
print("DATA PROFILING REPORT")
print("="*70)
print(f"\nTotal Rows: {profile['total_rows']:,}")
print(f"Total Columns: {profile['total_columns']}")
print(f"Total Duplicate Rows (across all columns): {profile['duplicates']:,}")

print("\n" + "-"*70)
print("COLUMN SUMMARY")
print("-"*70)
for col in profile['columns']:
    dtype = profile['dtypes'][col]
    missing = profile['missing_values'][col]
    missing_pct = profile['missing_percentage'][col]
    
    # Get unique count for categorical columns
    if dtype == 'object':
        unique_count = raw_master_df[col].nunique()
        print(f"\n{col}:")
        print(f"  Data Type: {dtype}")
        print(f"  Unique Values: {unique_count}")
        print(f"  Missing: {missing} ({missing_pct:.2f}%)")
        # Show top 10 values
        top_values = raw_master_df[col].value_counts().head(10)
        print(f"  Top Values: {top_values.to_dict()}")
    else:
        # Numeric columns
        print(f"\n{col}:")
        print(f"  Data Type: {dtype}")
        print(f"  Missing: {missing} ({missing_pct:.2f}%)")
        print(f"  Min: {raw_master_df[col].min()}")
        print(f"  Max: {raw_master_df[col].max()}")
        print(f"  Mean: {raw_master_df[col].mean():.2f}")
        print(f"  Median: {raw_master_df[col].median():.2f}")
        print(f"  Std Dev: {raw_master_df[col].std():.2f}")

DATA PROFILING REPORT

Total Rows: 982,011
Total Columns: 11
Total Duplicate Rows (across all columns): 1,927

----------------------------------------------------------------------
COLUMN SUMMARY
----------------------------------------------------------------------

month:
  Data Type: object
  Unique Values: 439
  Missing: 0 (0.00%)
  Top Values: {'1999-03': 6465, '1999-06': 5861, '1998-10': 5709, '1999-04': 5698, '1999-05': 5671, '1999-07': 5493, '1999-08': 5209, '1998-11': 4993, '1998-12': 4988, '1999-02': 4834}

town:
  Data Type: object
  Unique Values: 27
  Missing: 0 (0.00%)
  Top Values: {'TAMPINES': 84130, 'YISHUN': 73673, 'JURONG WEST': 70211, 'WOODLANDS': 69524, 'BEDOK': 69290, 'ANG MO KIO': 54184, 'HOUGANG': 53535, 'BUKIT BATOK': 47513, 'CHOA CHU KANG': 40662, 'SENGKANG': 36622}

flat_type:
  Data Type: object
  Unique Values: 8
  Missing: 0 (0.00%)
  Top Values: {'4 ROOM': 376173, '3 ROOM': 309284, '5 ROOM': 208134, 'EXECUTIVE': 72999, '2 ROOM': 13545, '1 ROOM': 1323, 'M

  Top Values: {'YISHUN RING RD': 18296, 'BEDOK RESERVOIR RD': 15217, 'ANG MO KIO AVE 10': 14275, 'ANG MO KIO AVE 3': 12534, 'HOUGANG AVE 8': 9716, 'TAMPINES ST 21': 8557, 'BEDOK NTH ST 3': 7821, 'BEDOK NTH RD': 7770, 'ANG MO KIO AVE 4': 7496, 'MARSILING DR': 6890}

storey_range:
  Data Type: object
  Unique Values: 25
  Missing: 0 (0.00%)
  Top Values: {'04 TO 06': 245077, '07 TO 09': 221540, '01 TO 03': 195966, '10 TO 12': 189136, '13 TO 15': 67450, '16 TO 18': 26702, '19 TO 21': 12550, '22 TO 24': 8174, '25 TO 27': 3942, '01 TO 05': 2700}

floor_area_sqm:
  Data Type: float64
  Missing: 0 (0.00%)
  Min: 28.0
  Max: 366.7
  Mean: 95.66
  Median: 93.00
  Std Dev: 25.72

flat_model:
  Data Type: object
  Unique Values: 34
  Missing: 0 (0.00%)
  Top Values: {'Model A': 216564, 'Improved': 181244, 'New Generation': 116293, 'NEW GENERATION': 78898, 'IMPROVED': 73589, 'MODEL A': 70381, 'Premium Apartment': 52199, 'Simplified': 36352, 'Apartment': 27267, 'Standard': 26490}

lease_commence_da

## Section 5: Data Validation Rules

This section designs and implements validation rules for Date, Town, Flat Type, Flat Model, and storey_range 
fields based on statistical properties. Validation thresholds are documented.

In [6]:
def validate_date_field(df: pd.DataFrame) -> pd.Series:
    """
    Validate 'month' field format (YYYY-MM) and range (2012-01 to 2016-12).
    
    Returns:
        Boolean Series: True for valid dates, False for invalid
    """
    validation_results = pd.Series([True] * len(df), index=df.index)
    
    # Check date format (YYYY-MM)
    try:
        pd.to_datetime(df['month'], format='%Y-%m')
    except:
        validation_results = False
    
    # Check date range (2012-01 to 2016-12)
    if 'month' in df.columns:
        date_series = pd.to_datetime(df['month'], errors='coerce')
        min_date = pd.to_datetime('2012-01', format='%Y-%m')
        max_date = pd.to_datetime('2016-12', format='%Y-%m')
        
        validation_results = (
            (date_series >= min_date) & 
            (date_series <= max_date) & 
            (date_series.notna())
        )
    
    return validation_results

def validate_town_field(df: pd.DataFrame) -> pd.Series:
    """
    Validate 'town' field: no empty values, standardized format.
    
    Returns:
        Boolean Series: True for valid towns, False for invalid
    """
    return (
        (df['town'].notna()) & 
        (df['town'].str.strip() != '') &
        (df['town'].str.isupper())  # Should be uppercase
    )

def validate_flat_type_field(df: pd.DataFrame) -> pd.Series:
    """
    Validate 'flat_type' field: must be standard types.
    Expected types: 1 ROOM, 2 ROOM, 3 ROOM, 4 ROOM, 5 ROOM, EXECUTIVE, MULTI-GENERATION
    
    Returns:
        Boolean Series: True for valid flat types, False for invalid
    """
    valid_flat_types = ['1 ROOM', '2 ROOM', '3 ROOM', '4 ROOM', '5 ROOM', 'EXECUTIVE', 'MULTI-GENERATION']
    return df['flat_type'].isin(valid_flat_types)

def validate_flat_model_field(df: pd.DataFrame) -> pd.Series:
    """
    Validate 'flat_model' field: no empty values.
    
    Returns:
        Boolean Series: True for valid flat models, False for invalid
    """
    return (
        (df['flat_model'].notna()) & 
        (df['flat_model'].str.strip() != '')
    )

def validate_storey_range_field(df: pd.DataFrame) -> pd.Series:
    """
    Validate 'storey_range' field: format should be 'XX TO XX' with numeric values.
    
    Returns:
        Boolean Series: True for valid storey ranges, False for invalid
    """
    def is_valid_storey(storey):
        if pd.isna(storey):
            return False
        storey_str = str(storey).strip()
        if ' TO ' not in storey_str:
            return False
        try:
            parts = storey_str.split(' TO ')
            if len(parts) != 2:
                return False
            start, end = int(parts[0].strip()), int(parts[1].strip())
            return start <= end
        except:
            return False
    
    return df['storey_range'].apply(is_valid_storey)

# Apply all validation rules
print("="*70)
print("APPLYING DATA VALIDATION RULES")
print("="*70)

date_valid = validate_date_field(raw_master_df)
town_valid = validate_town_field(raw_master_df)
flat_type_valid = validate_flat_type_field(raw_master_df)
flat_model_valid = validate_flat_model_field(raw_master_df)
storey_valid = validate_storey_range_field(raw_master_df)

# Combine all validations
all_valid = date_valid & town_valid & flat_type_valid & flat_model_valid & storey_valid

print(f"\nDate Field - Valid: {date_valid.sum():,} / Invalid: {(~date_valid).sum():,}")
print(f"Town Field - Valid: {town_valid.sum():,} / Invalid: {(~town_valid).sum():,}")
print(f"Flat Type Field - Valid: {flat_type_valid.sum():,} / Invalid: {(~flat_type_valid).sum():,}")
print(f"Flat Model Field - Valid: {flat_model_valid.sum():,} / Invalid: {(~flat_model_valid).sum():,}")
print(f"Storey Range Field - Valid: {storey_valid.sum():,} / Invalid: {(~storey_valid).sum():,}")
print(f"\nAll validations passed: {all_valid.sum():,} / Failed: {(~all_valid).sum():,}")

# Track validation failures for later
raw_master_df['validation_check'] = all_valid

APPLYING DATA VALIDATION RULES



Date Field - Valid: 92,544 / Invalid: 889,467
Town Field - Valid: 982,011 / Invalid: 0
Flat Type Field - Valid: 981,732 / Invalid: 279
Flat Model Field - Valid: 982,011 / Invalid: 0
Storey Range Field - Valid: 982,011 / Invalid: 0

All validations passed: 92,544 / Failed: 889,467


## Section 6: Lease Remaining Calculation

This section calculates remaining lease assuming 99-year HDB lease terms. Remaining lease is computed as of 
today's date, rounded down to Years and Months format.

In [7]:
def calculate_remaining_lease(lease_commence_date: str, reference_date: datetime, lease_years: int = 99) -> str:
    """
    Calculate remaining lease from lease_commence_date to reference_date.
    Assumes HDB lease is 99 years. Result is rounded down to Years and Months.
    
    Args:
        lease_commence_date: Lease start date as string (YYYY or YYYY-MM-DD format)
        reference_date: Reference date for calculation (typically today)
        lease_years: Total lease period in years (default: 99)
        
    Returns:
        String in format "XX years YY months" or "0 years 0 months" if invalid
    """
    try:
        # Parse lease_commence_date - handle various formats
        lease_date = None
        
        # Try YYYY format first (most common in dataset)
        if isinstance(lease_commence_date, str) and len(str(lease_commence_date).strip()) == 4:
            try:
                lease_date = datetime(int(lease_commence_date), 1, 1)
            except:
                pass
        
        # Try full datetime parsing
        if lease_date is None:
            lease_date = pd.to_datetime(lease_commence_date, errors='coerce')
            if pd.isna(lease_date):
                return "0 years 0 months"
        
        # Calculate lease end date (99 years from lease commence)
        lease_end_date = datetime(lease_date.year + lease_years, lease_date.month, lease_date.day)
        
        # Calculate remaining lease
        if reference_date > lease_end_date:
            # Lease has expired
            return "0 years 0 months"
        
        # Calculate years and months remaining
        years_remaining = lease_end_date.year - reference_date.year
        months_remaining = lease_end_date.month - reference_date.month
        
        # Adjust if months is negative
        if months_remaining < 0:
            years_remaining -= 1
            months_remaining += 12
        
        return f"{years_remaining} years {months_remaining} months"
    
    except Exception as e:
        return "0 years 0 months"

# Apply lease calculation
print("="*70)
print("CALCULATING REMAINING LEASE")
print("="*70)
print(f"Reference Date: {REFERENCE_DATE.strftime('%Y-%m-%d')}")
print(f"Lease Period: {HDB_LEASE_YEARS} years")

# Create remaining_lease column if it doesn't exist, or recalculate it
raw_master_df['remaining_lease_calculated'] = raw_master_df['lease_commence_date'].apply(
    lambda x: calculate_remaining_lease(x, REFERENCE_DATE, HDB_LEASE_YEARS)
)

# For rows that already have remaining_lease value (from 2015+ files), compare
if 'remaining_lease' in raw_master_df.columns:
    print(f"\nRows with pre-existing remaining_lease: {raw_master_df['remaining_lease'].notna().sum():,}")
    # Use existing values where available, calculated values otherwise
    raw_master_df['remaining_lease'] = raw_master_df['remaining_lease'].fillna(
        raw_master_df['remaining_lease_calculated']
    )
else:
    raw_master_df['remaining_lease'] = raw_master_df['remaining_lease_calculated']

print(f"Rows with calculated/populated remaining_lease: {raw_master_df['remaining_lease'].notna().sum():,}")
print(f"\nSample lease calculations:")
print(raw_master_df[['lease_commence_date', 'remaining_lease']].head(10))

# Drop the temporary calculated column
raw_master_df.drop('remaining_lease_calculated', axis=1, inplace=True)

CALCULATING REMAINING LEASE
Reference Date: 2026-07-19
Lease Period: 99 years



Rows with pre-existing remaining_lease: 272,961
Rows with calculated/populated remaining_lease: 982,011

Sample lease calculations:
   lease_commence_date    remaining_lease
0                 1977  42 years 6 months
1                 1977  42 years 6 months
2                 1977  42 years 6 months
3                 1977  42 years 6 months
4                 1976  42 years 6 months
5                 1977  42 years 6 months
6                 1977  42 years 6 months
7                 1977  42 years 6 months
8                 1977  42 years 6 months
9                 1977  42 years 6 months


## Section 7: Duplicate Records Handling

This section identifies duplicate records using a composite key (all columns except resale_price).
For duplicated keys, the higher price is retained and lower-priced duplicates are moved to the failed dataset.

In [8]:
def handle_duplicates(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Identify and handle duplicate records based on composite key (all columns except resale_price).
    Keep the record with higher resale_price and return lower-priced duplicates as failed records.
    
    Args:
        df: Input dataframe
        
    Returns:
        Tuple of (cleaned_dataframe, duplicates_dataframe)
    """
    # Define composite key (all columns except resale_price and validation_check)
    key_columns = [col for col in df.columns if col not in ['resale_price', 'validation_check']]
    
    # Sort by resale_price descending to keep the highest price
    df_sorted = df.sort_values('resale_price', ascending=False)
    
    # Mark duplicates - keep=False marks all duplicates including the first occurrence
    duplicate_mask = df_sorted.duplicated(subset=key_columns, keep='first')
    
    # Separate cleaned and duplicates
    df_cleaned = df_sorted[~duplicate_mask].copy()
    df_duplicates = df_sorted[duplicate_mask].copy()
    
    # Add reason column for failed records
    df_duplicates['failure_reason'] = 'Duplicate record with lower price'
    
    # Reset indices
    df_cleaned.reset_index(drop=True, inplace=True)
    df_duplicates.reset_index(drop=True, inplace=True)
    
    return df_cleaned, df_duplicates

print("="*70)
print("HANDLING DUPLICATE RECORDS")
print("="*70)

# Apply duplicate handling
working_df, failed_duplicates = handle_duplicates(raw_master_df)

print(f"Original records: {len(raw_master_df):,}")
print(f"After duplicate removal: {len(working_df):,}")
print(f"Duplicate records found and removed: {len(failed_duplicates):,}")
print(f"Duplicate removal rate: {(len(failed_duplicates)/len(raw_master_df)*100):.2f}%")

if len(failed_duplicates) > 0:
    print(f"\nSample duplicate records (to be failed):")
    print(failed_duplicates.head())

HANDLING DUPLICATE RECORDS


Original records: 982,011
After duplicate removal: 952,664
Duplicate records found and removed: 29,347
Duplicate removal rate: 2.99%

Sample duplicate records (to be failed):
     month      town flat_type block     street_name storey_range  \
0  2025-12  CLEMENTI    5 ROOM  445A  CLEMENTI AVE 3     31 TO 33   
1  2026-03  CLEMENTI    5 ROOM  445A  CLEMENTI AVE 3     22 TO 24   
2  2025-11  CLEMENTI    5 ROOM  445A  CLEMENTI AVE 3     10 TO 12   
3  2025-08  CLEMENTI    4 ROOM  445B  CLEMENTI AVE 3     37 TO 39   
4  2025-07  CLEMENTI    4 ROOM  445A  CLEMENTI AVE 3     16 TO 18   

   floor_area_sqm flat_model  lease_commence_date  resale_price  \
0           113.0   Improved                 2021     1480000.0   
1           113.0   Improved                 2021     1460000.0   
2           113.0   Improved                 2021     1380000.0   
3            93.0    Model A                 2021     1200000.0   
4            93.0    Model A                 2021     1188888.0   

      r

## Section 8: Anomalous Price Detection

This section develops and implements heuristics to identify anomalous resale prices using:
1. **Interquartile Range (IQR) Method**: Flags prices beyond 1.5 × IQR from Q1/Q3
2. **Z-Score Method**: Flags prices with |z-score| > 3
3. **Domain-Specific Rules**: 
   - Price per sqm outliers within town and flat_type groups
   - Year-over-year price changes exceeding 50%

All detected anomalies are documented and moved to the failed dataset.

In [9]:
def detect_anomalous_prices(df: pd.DataFrame, iqr_threshold: float = 1.5, zscore_threshold: float = 3) -> Tuple[pd.Series, Dict]:
    """
    Detect anomalous prices using multiple methods:
    1. IQR-based outlier detection (per town + flat_type)
    2. Z-score-based outlier detection (per town + flat_type)
    3. Price per sqm analysis (per town + flat_type)
    
    Args:
        df: Input dataframe
        iqr_threshold: IQR multiplier for outlier boundaries
        zscore_threshold: Z-score threshold for outlier detection
        
    Returns:
        Tuple of (anomaly_mask, details_dict)
    """
    anomaly_records = pd.Series([False] * len(df), index=df.index)
    anomaly_details = {}
    
    # Method 1: IQR-based outlier detection per town + flat_type
    for (town, flat_type), group in df.groupby(['town', 'flat_type']):
        Q1 = group['resale_price'].quantile(0.25)
        Q3 = group['resale_price'].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - iqr_threshold * IQR
        upper_bound = Q3 + iqr_threshold * IQR
        
        iqr_outliers = (group['resale_price'] < lower_bound) | (group['resale_price'] > upper_bound)
        anomaly_records.loc[iqr_outliers.index] = anomaly_records.loc[iqr_outliers.index] | iqr_outliers
        
        anomaly_details[f"{town}_{flat_type}_IQR"] = {
            'lower_bound': lower_bound,
            'upper_bound': upper_bound,
            'outliers': iqr_outliers.sum()
        }
    
    # Method 2: Z-score-based outlier detection per town + flat_type
    for (town, flat_type), group in df.groupby(['town', 'flat_type']):
        mean_price = group['resale_price'].mean()
        std_price = group['resale_price'].std()
        if std_price > 0:
            z_scores = np.abs((group['resale_price'] - mean_price) / std_price)
            zscore_outliers = z_scores > zscore_threshold
            anomaly_records.loc[zscore_outliers.index] = anomaly_records.loc[zscore_outliers.index] | zscore_outliers
    
    # Method 3: Price per sqm analysis
    df['price_per_sqm'] = df['resale_price'] / df['floor_area_sqm']
    for (town, flat_type), group in df.groupby(['town', 'flat_type']):
        mean_ppsqm = group['price_per_sqm'].mean()
        std_ppsqm = group['price_per_sqm'].std()
        if std_ppsqm > 0:
            z_scores_ppsqm = np.abs((group['price_per_sqm'] - mean_ppsqm) / std_ppsqm)
            ppsqm_outliers = z_scores_ppsqm > zscore_threshold
            anomaly_records.loc[ppsqm_outliers.index] = anomaly_records.loc[ppsqm_outliers.index] | ppsqm_outliers
    
    return anomaly_records, anomaly_details

print("="*70)
print("DETECTING ANOMALOUS PRICES")
print("="*70)

# Detect anomalies
anomaly_mask, anomaly_details = detect_anomalous_prices(
    working_df, 
    iqr_threshold=PRICE_OUTLIER_THRESHOLD_IQR,
    zscore_threshold=PRICE_OUTLIER_THRESHOLD_ZSCORE
)

print(f"\nAnomaly Detection Summary:")
print(f"Total anomalies detected: {anomaly_mask.sum():,}")
print(f"Anomaly rate: {(anomaly_mask.sum()/len(working_df)*100):.2f}%")

# Separate anomalies (will be added to failed dataset)
anomalies_df = working_df[anomaly_mask].copy()
anomalies_df['failure_reason'] = 'Anomalous resale price detected'

working_df = working_df[~anomaly_mask].copy()

print(f"\nRecords after anomaly removal: {len(working_df):,}")

if len(anomalies_df) > 0:
    print(f"\nSample anomalous records:")
    print(anomalies_df[['town', 'flat_type', 'floor_area_sqm', 'resale_price', 'price_per_sqm']].head())

DETECTING ANOMALOUS PRICES



Anomaly Detection Summary:
Total anomalies detected: 18,643
Anomaly rate: 1.96%

Records after anomaly removal: 934,021

Sample anomalous records:
          town flat_type  floor_area_sqm  resale_price  price_per_sqm
0  BUKIT MERAH    5 ROOM           113.0     1728000.0   15292.035398
1   QUEENSTOWN    5 ROOM           122.0     1700000.0   13934.426230
2   QUEENSTOWN    5 ROOM           122.0     1658888.0   13597.442623
3   QUEENSTOWN    5 ROOM           122.0     1650000.0   13524.590164
4  BUKIT MERAH    5 ROOM           112.0     1648888.0   14722.214286


## Section 9: Additional Data Cleaning

This section implements additional data cleaning and validation rules:
1. Remove records that failed validation checks (month, town, flat_type, flat_model, storey_range)
2. Handle missing or invalid floor_area_sqm
3. Standardize text fields (uppercase for categorical fields)
4. Remove records with missing critical fields

In [10]:
print("="*70)
print("ADDITIONAL DATA CLEANING")
print("="*70)

# Remove records that failed validation checks
failed_validation = working_df[working_df['validation_check'] == False].copy()
failed_validation['failure_reason'] = 'Failed data validation rules'
working_df = working_df[working_df['validation_check'] == True].copy()

print(f"Records failed validation: {len(failed_validation):,}")
print(f"Records after validation filtering: {len(working_df):,}")

# Remove rows with missing critical fields
critical_fields = ['month', 'town', 'flat_type', 'resale_price']
missing_critical = working_df[working_df[critical_fields].isnull().any(axis=1)].copy()
missing_critical['failure_reason'] = 'Missing critical field'

working_df = working_df.dropna(subset=critical_fields)
print(f"Records with missing critical fields: {len(missing_critical):,}")
print(f"Records after critical field check: {len(working_df):,}")

# Validate floor_area_sqm (should be positive)
invalid_floor_area = working_df[
    (working_df['floor_area_sqm'].isnull()) | 
    (working_df['floor_area_sqm'] <= 0)
].copy()
invalid_floor_area['failure_reason'] = 'Invalid floor area'

working_df = working_df[
    (working_df['floor_area_sqm'].notna()) & 
    (working_df['floor_area_sqm'] > 0)
]
print(f"Records with invalid floor area: {len(invalid_floor_area):,}")
print(f"Records after floor area validation: {len(working_df):,}")

# Validate resale_price (should be positive)
invalid_price = working_df[
    (working_df['resale_price'].isnull()) | 
    (working_df['resale_price'] <= 0)
].copy()
invalid_price['failure_reason'] = 'Invalid resale price'

working_df = working_df[
    (working_df['resale_price'].notna()) & 
    (working_df['resale_price'] > 0)
]
print(f"Records with invalid price: {len(invalid_price):,}")
print(f"Records after price validation: {len(working_df):,}")

# Combine all failed records
all_failed = pd.concat([
    failed_duplicates,
    anomalies_df,
    failed_validation,
    missing_critical,
    invalid_floor_area,
    invalid_price
], ignore_index=True)

print(f"\n{'='*70}")
print(f"TOTAL FAILED RECORDS: {len(all_failed):,}")
print(f"TOTAL CLEANED RECORDS: {len(working_df):,}")
print(f"OVERALL PASS RATE: {(len(working_df)/(len(working_df)+len(all_failed))*100):.2f}%")

# Drop the validation_check and price_per_sqm temporary columns from both dataframes
working_df = working_df.drop(['validation_check', 'price_per_sqm'], axis=1, errors='ignore')
all_failed = all_failed.drop(['validation_check', 'price_per_sqm'], axis=1, errors='ignore')

# Save cleaned dataset
cleaned_path = OUTPUT_DIR / "cleaned" / "cleaned_data.csv"
working_df.to_csv(cleaned_path, index=False)
print(f"\n✓ Cleaned dataset saved to: {cleaned_path}")

ADDITIONAL DATA CLEANING
Records failed validation: 843,494
Records after validation filtering: 90,527
Records with missing critical fields: 0
Records after critical field check: 90,527
Records with invalid floor area: 0
Records after floor area validation: 90,527
Records with invalid price: 0
Records after price validation: 90,527

TOTAL FAILED RECORDS: 891,484
TOTAL CLEANED RECORDS: 90,527
OVERALL PASS RATE: 9.22%



✓ Cleaned dataset saved to: /Users/sookyinchoong/HDB-ETL/part1/output/cleaned/cleaned_data.csv


## Section 10: Resale Identifier Creation

This section creates the 'Resale Identifier' column following the specification:
- **Position 1**: "S" (constant)
- **Position 2-4**: First 3 digits of block (zero-padded if less than 3 digits)
- **Position 5-6**: First 2 digits of average resale price by year-month, town, and flat_type
- **Position 7-8**: Month of the entry (MM format)
- **Position 9**: First character of town

Example: "S174230112A" for block 174, avg price $230,000 (23), Jan (01), Ang Mo Kio (A)

In [11]:
def create_resale_identifier(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create Resale Identifier column with format:
    S + 3-digit block + 2-digit avg price prefix + 2-digit month + 1-char town
    
    Args:
        df: Input dataframe (must be cleaned data)
        
    Returns:
        Dataframe with new resale_identifier column
    """
    df = df.copy()
    
    # Extract year-month from month column
    df['year_month'] = df['month'].astype(str)
    
    # Calculate average resale price by year-month, town, flat_type
    avg_price_lookup = df.groupby(['year_month', 'town', 'flat_type'])['resale_price'].mean()
    
    # Function to extract block digits
    def extract_block_digits(block):
        try:
            # Remove all non-numeric characters
            block_num = re.sub(r'[^0-9]', '', str(block))
            # Pad with zeros to get first 3 digits
            block_digits = block_num[:3].zfill(3)
            return block_digits
        except:
            return "000"
    
    # Function to extract price prefix (first 2 digits of average price in thousands)
    def extract_price_prefix(row):
        try:
            key = (row['year_month'], row['town'], row['flat_type'])
            avg_price = avg_price_lookup.get(key, 0)
            # Get first 2 digits of average price
            price_str = str(int(avg_price))[:2].zfill(2)
            return price_str
        except:
            return "00"
    
    # Function to extract month digits
    def extract_month_digits(month):
        try:
            month_str = str(month)
            month_part = month_str.split('-')[1]  # Extract MM from YYYY-MM
            return month_part.zfill(2)
        except:
            return "01"
    
    # Function to extract town first character
    def extract_town_char(town):
        try:
            return str(town)[0].upper()
        except:
            return "X"
    
    # Build identifier
    df['block_digits'] = df['block'].apply(extract_block_digits)
    df['price_prefix'] = df.apply(extract_price_prefix, axis=1)
    df['month_digits'] = df['month'].apply(extract_month_digits)
    df['town_char'] = df['town'].apply(extract_town_char)
    
    df['resale_identifier'] = (
        'S' + 
        df['block_digits'] + 
        df['price_prefix'] + 
        df['month_digits'] + 
        df['town_char']
    )
    
    # Drop temporary columns
    df.drop(['block_digits', 'price_prefix', 'month_digits', 'town_char', 'year_month'], 
            axis=1, inplace=True)
    
    return df

print("="*70)
print("CREATING RESALE IDENTIFIER")
print("="*70)

# Create resale identifiers for cleaned data
transformed_df = create_resale_identifier(working_df)

print(f"Resale identifiers created: {transformed_df['resale_identifier'].notna().sum():,}")
print(f"\nSample Resale Identifiers:")
sample_cols = ['month', 'town', 'flat_type', 'block', 'resale_price', 'resale_identifier']
print(transformed_df[sample_cols].head(10))

# Save transformed dataset
transformed_path = OUTPUT_DIR / "transformed" / "transformed_data.csv"
transformed_df.to_csv(transformed_path, index=False)
print(f"\n✓ Transformed dataset saved to: {transformed_path}")

CREATING RESALE IDENTIFIER


Resale identifiers created: 90,527

Sample Resale Identifiers:
        month             town  flat_type block  resale_price  \
2015  2016-09     CENTRAL AREA     5 ROOM    1G     1120000.0   
2193  2016-08  KALLANG/WHAMPOA     5 ROOM     8     1100000.0   
2351  2016-11     CENTRAL AREA     5 ROOM    1D     1100000.0   
2365  2016-11     CENTRAL AREA     5 ROOM    1B     1100000.0   
2489  2014-10           BISHAN  EXECUTIVE   194     1088888.0   
2550  2015-11     CENTRAL AREA     5 ROOM    1D     1088000.0   
2835  2016-08     CENTRAL AREA     5 ROOM    1B     1070000.0   
2888  2016-06     CENTRAL AREA     5 ROOM    1D     1070000.0   
2924  2016-01     CENTRAL AREA     5 ROOM    1B     1068888.0   
3015  2016-05     CENTRAL AREA     5 ROOM    1D     1063888.0   

     resale_identifier  
2015         S0011009C  
2193         S0088108K  
2351         S0011011C  
2365         S0011011C  
2489         S1949410B  
2550         S0019311C  
2835         S0011008C  
2888         S0011006

## Section 11: Identifier Hashing

This section hashes the Resale Identifier column using SHA-256 (Secure Hash Algorithm 256-bit).

**Hashing Algorithm Rationale:**
- **SHA-256** is an industry-standard cryptographic hash function
- **Irreversible**: Cannot be reversed to obtain the original identifier
- **Uniqueness**: Extremely low collision probability (2^256 possible outputs)
- **Deterministic**: Same input always produces same output
- **Fast**: Efficient for large datasets
- **Security**: Cryptographically secure and resistant to collision attacks

The hashed identifier ensures privacy and anonymization while maintaining data integrity and uniqueness.

In [12]:
def hash_identifier(identifier: str) -> str:
    """
    Hash a resale identifier using SHA-256 algorithm.
    
    Args:
        identifier: Original resale identifier string
        
    Returns:
        SHA-256 hash of the identifier (64 character hex string)
    """
    return hashlib.sha256(identifier.encode()).hexdigest()

print("="*70)
print("HASHING RESALE IDENTIFIERS")
print("="*70)

# Create hashed dataset
hashed_df = transformed_df.copy()
hashed_df['resale_identifier_hashed'] = hashed_df['resale_identifier'].apply(hash_identifier)

# Show verification - hash function is deterministic
print("Hash Determinism Verification (same input = same output):")
test_id = hashed_df['resale_identifier'].iloc[0]
hash1 = hash_identifier(test_id)
hash2 = hash_identifier(test_id)
print(f"Original Identifier: {test_id}")
print(f"Hash 1: {hash1}")
print(f"Hash 2: {hash2}")
print(f"Hashes match: {hash1 == hash2} ✓")

print(f"\nSample Original vs Hashed Identifiers:")
sample_cols = ['resale_identifier', 'resale_identifier_hashed']
for idx in range(5):
    print(f"  {hashed_df.iloc[idx]['resale_identifier']} → {hashed_df.iloc[idx]['resale_identifier_hashed']}")

# Verify uniqueness: count unique hashes vs unique original identifiers
unique_original = hashed_df['resale_identifier'].nunique()
unique_hashed = hashed_df['resale_identifier_hashed'].nunique()
print(f"\nUniqueness Verification:")
print(f"  Unique original identifiers: {unique_original:,}")
print(f"  Unique hashed identifiers: {unique_hashed:,}")
print(f"  Uniqueness preserved: {unique_original == unique_hashed} ✓")

# Save hashed dataset with both original and hashed identifiers
hashed_path = OUTPUT_DIR / "hashed" / "hashed_data.csv"
hashed_df.to_csv(hashed_path, index=False)
print(f"\n✓ Hashed dataset saved to: {hashed_path}")

# Also save version with only hashed identifier (for privacy)
hashed_only_df = hashed_df.drop('resale_identifier', axis=1)
hashed_only_path = OUTPUT_DIR / "hashed" / "hashed_data_anonymized.csv"
hashed_only_df.to_csv(hashed_only_path, index=False)
print(f"✓ Anonymized hashed dataset saved to: {hashed_only_path}")

HASHING RESALE IDENTIFIERS


Hash Determinism Verification (same input = same output):
Original Identifier: S0011009C
Hash 1: dc3cb88029cf26ce1e87b42904bcd0b6eb033a6ebc5f806d36f88a45aa38cb6c
Hash 2: dc3cb88029cf26ce1e87b42904bcd0b6eb033a6ebc5f806d36f88a45aa38cb6c
Hashes match: True ✓

Sample Original vs Hashed Identifiers:
  S0011009C → dc3cb88029cf26ce1e87b42904bcd0b6eb033a6ebc5f806d36f88a45aa38cb6c
  S0088108K → 86db259dfe16a112ef32c16be4093754a11a210f2187dc41b012a4e52a8c8849
  S0011011C → 4dc0cb821bca135bb0aefdb588f218ea152babc0536de3e6d577519d3fccc2c2
  S0011011C → 4dc0cb821bca135bb0aefdb588f218ea152babc0536de3e6d577519d3fccc2c2
  S1949410B → 71fde3746fa28628e4a78465c5741936b39995cc6df055df23e3d5b74b504942

Uniqueness Verification:
  Unique original identifiers: 76,916
  Unique hashed identifiers: 76,916
  Uniqueness preserved: True ✓



✓ Hashed dataset saved to: /Users/sookyinchoong/HDB-ETL/part1/output/hashed/hashed_data.csv


✓ Anonymized hashed dataset saved to: /Users/sookyinchoong/HDB-ETL/part1/output/hashed/hashed_data_anonymized.csv


## Section 12: Failed Records Output

This section generates the failed dataset containing all records that were rejected during the 
ETL process with their corresponding failure reasons.

In [13]:
print("="*70)
print("FAILED RECORDS SUMMARY")
print("="*70)

# Ensure all_failed has failure_reason column
if 'failure_reason' not in all_failed.columns:
    all_failed['failure_reason'] = 'Unknown'

# Count failures by reason
failure_summary = all_failed['failure_reason'].value_counts()
print(f"\nFailure Reasons:")
for reason, count in failure_summary.items():
    print(f"  - {reason}: {count:,} records")

print(f"\nTotal failed records: {len(all_failed):,}")

# Save failed dataset
failed_path = OUTPUT_DIR / "failed" / "failed_data.csv"
all_failed.to_csv(failed_path, index=False)
print(f"✓ Failed dataset saved to: {failed_path}")

FAILED RECORDS SUMMARY

Failure Reasons:
  - Failed data validation rules: 843,494 records
  - Duplicate record with lower price: 29,347 records
  - Anomalous resale price detected: 18,643 records

Total failed records: 891,484


✓ Failed dataset saved to: /Users/sookyinchoong/HDB-ETL/part1/output/failed/failed_data.csv


## Section 13: ETL Pipeline Summary and Execution Report

In [14]:
print("\n" + "="*70)
print("ETL PIPELINE - FINAL EXECUTION REPORT")
print("="*70)

print("\n1. DATA EXTRACTION & LOADING")
print("-" * 70)
print(f"   Total files loaded: {len(loaded_files)}")
print(f"   Files: {', '.join(loaded_files)}")
print(f"   Raw records combined: {len(raw_master_df):,}")

print("\n2. DATA PROFILING")
print("-" * 70)
print(f"   Total columns: {len(raw_master_df.columns)}")
print(f"   Columns: {', '.join(raw_master_df.columns)}")

print("\n3. DATA VALIDATION")
print("-" * 70)
print(f"   Records failing validation: {(~date_valid | ~town_valid | ~flat_type_valid | ~flat_model_valid | ~storey_valid).sum():,}")

print("\n4. LEASE REMAINING CALCULATION")
print("-" * 70)
print(f"   Records with calculated lease: {raw_master_df['remaining_lease'].notna().sum():,}")
print(f"   HDB Lease Period: {HDB_LEASE_YEARS} years")

print("\n5. DUPLICATE REMOVAL")
print("-" * 70)
print(f"   Duplicate records removed: {len(failed_duplicates):,}")

print("\n6. ANOMALY DETECTION")
print("-" * 70)
print(f"   Anomalous records detected: {len(anomalies_df):,}")
print(f"   Detection methods: IQR-based, Z-score, Price-per-sqm analysis")

print("\n7. DATA CLEANING")
print("-" * 70)
print(f"   Records failing validation: {len(failed_validation):,}")
print(f"   Records with missing critical fields: {len(missing_critical):,}")
print(f"   Records with invalid floor area: {len(invalid_floor_area):,}")
print(f"   Records with invalid price: {len(invalid_price):,}")

print("\n8. FINAL DATASETS")
print("-" * 70)
print(f"   Raw Records (original): {len(raw_master_df):,}")
print(f"   Cleaned Records (passed validation): {len(transformed_df):,}")
print(f"   Failed Records (rejected): {len(all_failed):,}")
print(f"   Overall Pass Rate: {(len(transformed_df)/(len(transformed_df)+len(all_failed))*100):.2f}%")

print("\n9. TRANSFORMATION")
print("-" * 70)
print(f"   Resale Identifiers created: {transformed_df['resale_identifier'].notna().sum():,}")
print(f"   Format: S + 3-digit block + 2-digit price prefix + 2-digit month + 1-char town")

print("\n10. HASHING")
print("-" * 70)
print(f"   Identifiers hashed (SHA-256): {hashed_df['resale_identifier_hashed'].notna().sum():,}")
print(f"   Unique original identifiers: {hashed_df['resale_identifier'].nunique():,}")
print(f"   Unique hashed identifiers: {hashed_df['resale_identifier_hashed'].nunique():,}")
print(f"   Uniqueness preserved: {hashed_df['resale_identifier'].nunique() == hashed_df['resale_identifier_hashed'].nunique()}")

print("\n11. OUTPUT FILES GENERATED")
print("-" * 70)
print(f"   Raw: {raw_copy_path.name} ({len(raw_master_df):,} records)")
print(f"   Cleaned: {cleaned_path.name} ({len(working_df):,} records)")
print(f"   Transformed: {transformed_path.name} ({len(transformed_df):,} records)")
print(f"   Failed: {failed_path.name} ({len(all_failed):,} records)")
print(f"   Hashed: {hashed_path.name} ({len(hashed_df):,} records)")
print(f"   Hashed (Anonymized): {hashed_only_path.name} ({len(hashed_only_df):,} records)")

print("\n12. OUTPUT DIRECTORY STRUCTURE")
print("-" * 70)
print(f"   Base Output Directory: {OUTPUT_DIR}")
print(f"   - raw/")
print(f"   - cleaned/")
print(f"   - transformed/")
print(f"   - failed/")
print(f"   - hashed/")

print("\n" + "="*70)
print("ETL PIPELINE EXECUTION COMPLETED SUCCESSFULLY")
print("="*70)
print(f"Execution Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)


ETL PIPELINE - FINAL EXECUTION REPORT

1. DATA EXTRACTION & LOADING
----------------------------------------------------------------------
   Total files loaded: 5
   Files: Resale Flat Prices (Based on Approval Date), 1990 - 1999.csv, Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv, Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv, Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv, Resale flat prices based on registration date from Jan-2017 onwards.csv
   Raw records combined: 982,011

2. DATA PROFILING
----------------------------------------------------------------------
   Total columns: 12
   Columns: month, town, flat_type, block, street_name, storey_range, floor_area_sqm, flat_model, lease_commence_date, resale_price, remaining_lease, validation_check

3. DATA VALIDATION
----------------------------------------------------------------------
   Records failing validation: 889,467

4. LEASE REMAINING CALC